# Latent physical-organization ratio

This notebook computes a single diagnostic per autoencoder setup: **does this
setup's latent space encode cross-climate physical structure throughout its
whole representation, or only along a lucky handful of directions?**

It is a standalone follow-up to `CMIP_analysis_latent_invariant.ipynb` (same
project, same data). That notebook found, per setup, the single BEST
invariant direction in the latent (via a generalized eigenvalue
decomposition) and used it to hunt for interpretable physical formulas. This
notebook asks a different, complementary question: instead of the single
best direction, what does a *typical* (random) direction look like? If a
setup's whole latent is well-organized physically, random directions should
*also* look decent on average -- not just the one special axis.

Setups covered: **CERA, Baseline, SWDN** (all restricted to the first 48
"aligned" dimensions of their respective 64D latents) and **ClimaX** (its
full 64D latent, no truncation).


## Part 0 - Configuration and imports

Same run configuration as `CMIP_analysis_latent_invariant.ipynb` (same
`num_sample`, autoencoder variant, and split fractions) so that this notebook
loads the *exact same* underlying data, splits, and latent files -- this is
what makes its `pair_results`, anchors, etc. directly comparable to that
notebook's results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import json
from pathlib import Path
import pickle
from sklearn.neighbors import NearestNeighbors


In [ ]:
num_sample = 2500000
chosen_autoencoder_type = "CNN"   # choose between "MLP" and "CNN"
inv_alignment_method = "swd"      # choose between "swd" and "adversarial"
variable = "pr"                   # variable excluded from the 15 input variables (prediction target upstream)
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.45
cera_lambda_pred = 0.1


## Part 1 - Data loading

**What we need, and why we can skip most of the original notebook's Part 0/1:**

This analysis never builds a raw-variable *formula* -- everything here is
computed directly on latent vectors. The only reason we need the raw grid
data at all is to build the **RawData baseline** (a nearest-neighbour
pairing based on the 15 raw input variables, used as the "trivial matching"
reference the whole ratio is normalized against). That baseline only needs
the 15 raw variables in standardized form -- **not** the 25 derived physical
invariants (RH, lapse rate, shear, anomalies, ...) that
`CMIP_analysis_latent_invariant.ipynb` computes for its own formula search.
Skipping that computation (which involves a slow per-sample climatology loop)
is the main simplification here; loading the raw grids themselves is
unfortunately still required and is the heaviest step in this notebook
(same cost as the original notebook's Part 0).


In [ ]:
# ── Raw data (needed only for the 15 input variables, in standardized form) ───
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
random_seed = int(run_cfg["random_seed"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
assert n_lat * n_lon == grid_points_per_patch

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

selected_variables_full = list(run_cfg["selected_variables"])
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_var_index = selected_variables_full.index(variable)
variable_col_start = variable_var_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]
n_input_variables = len(selected_variables)

del features_by_climate_full   # free the heavy full-variable arrays, we only kept the 15-var slice

print(f"Loaded {len(climate_order)} climates. Raw input variables (excluding '{variable}'): {selected_variables}")
for c in climate_order:
    print(f"  {c}: {features_by_climate[c].shape}")


In [ ]:
# ── Latent representations (only the 4 setups we need) ────────────────────────
latent_root = Path("/glade/work/tsalin/CMIP/latent_representations")
if not latent_root.exists():
    raise FileNotFoundError(f"Latent representations directory not found: {latent_root}")


def _load_latent_payload(file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"Latent representations file not found: {file_path}")
    with open(file_path, "rb") as handle:
        payload = pickle.load(handle)
    latent_by_climate = {}
    metadata_by_climate_latent = {}
    for climate, entry in payload.items():
        if "latent" not in entry:
            raise KeyError(f"Missing 'latent' entry for climate '{climate}' in {file_path}")
        latent_by_climate[climate] = np.asarray(entry["latent"])
        metadata_by_climate_latent[climate] = pd.DataFrame(entry["metadata"]).reset_index(drop=True)
    return latent_by_climate, metadata_by_climate_latent


_exp5_latent_dir = latent_root / "latent_representations_exp_5"

exp5_cera_latent_file = _exp5_latent_dir / (
    f"cera_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_"
    f"{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_latent_representations_df.pkl"
)
exp5_baseline2_latent_file = _exp5_latent_dir / (
    f"baseline_CERA_noalign_ns{num_sample}_{chosen_autoencoder_type}_{variable}_"
    f"{val_fraction}_{test_fraction}_{cera_lambda_pred}_latent_representations_df.pkl"
)
exp5_baseline_climax_latent_file = _exp5_latent_dir / (
    f"baseline_ClimaX_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"
)
exp5_cera_swdn_latent_file = _exp5_latent_dir / (
    f"cera_ns{num_sample}_{chosen_autoencoder_type}_swdn_{variable}_"
    f"{val_fraction}_{test_fraction}_0.85_0.1_latent_representations_df.pkl"
)

exp5_cera_latent_test_by_climate, exp5_cera_latent_test_metadata_by_climate = _load_latent_payload(exp5_cera_latent_file)
exp5_baseline2_latent_test_by_climate, exp5_baseline2_latent_test_metadata_by_climate = _load_latent_payload(exp5_baseline2_latent_file)
exp5_baseline_climax_latent_test_by_climate, exp5_baseline_climax_latent_test_metadata_by_climate = _load_latent_payload(exp5_baseline_climax_latent_file)
exp5_cera_swdn_latent_test_by_climate, exp5_cera_swdn_latent_test_metadata_by_climate = _load_latent_payload(exp5_cera_swdn_latent_file)

latent_test_sets = {
    "exp5_cera": {
        "latent_test_by_climate": exp5_cera_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_latent_file,
    },
    "exp5_baseline2": {
        "latent_test_by_climate": exp5_baseline2_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_baseline2_latent_test_metadata_by_climate,
        "latent_file": exp5_baseline2_latent_file,
    },
    "exp5_baseline_climax": {
        "latent_test_by_climate": exp5_baseline_climax_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_baseline_climax_latent_test_metadata_by_climate,
        "latent_file": exp5_baseline_climax_latent_file,
    },
    "exp5_cera_swdn": {
        "latent_test_by_climate": exp5_cera_swdn_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_swdn_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_swdn_latent_file,
    },
}

# setup_name -> (latent_key, slice_dim). CERA/Baseline/SWDN keep only the first
# 48 "aligned" dimensions of their 64D latent; ClimaX uses its full 64D latent.
PAIR_LATENT_KEYS = {
    "CERA":     ("exp5_cera", 48),
    "Baseline": ("exp5_baseline2", 48),
    "SWDN":     ("exp5_cera_swdn", 48),
    "ClimaX":   ("exp5_baseline_climax", None),
}
SUBSPACE_SETUPS = ["CERA", "Baseline", "SWDN", "ClimaX"]

print("Loaded latent representations:")
for latent_name, latent_payload in latent_test_sets.items():
    print(f"  {latent_name}: {latent_payload['latent_file'].name}")


## Part 2 - Train/test splits and raw-variable standardization

Same split procedure as the original notebook (`build_split_indices`, same
seed), so `ae_split_indices` matches exactly. SSP ("eval-only") climates are
immediately truncated to their test split to save memory, exactly like the
"RAM Reduction" step there.

We then standardize the 15 raw input variables (mean/std fit on the
historical train split only) -- this is the *only* preprocessing needed here,
since we don't use the 25 derived physical invariants at all in this
notebook.


In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    split_indices = {}
    rng = np.random.default_rng(seed)
    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)
        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)
        split_indices[climate] = {
            "train": indices[:n_train],
            "val":   indices[n_train:n_train + n_val],
            "test":  indices[n_train + n_val:],
        }
    return split_indices


ae_split_indices = build_split_indices(
    features_by_climate, val_fraction=val_fraction, test_fraction=test_fraction, seed=random_seed,
)

baseline_train_climates = ["historical"]
_eval_only_climates = [c for c in climate_order if c not in baseline_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        features_by_climate[c] = features_by_climate[c][_idx]
        metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt: {_eval_only_climates} truncated to their test split.")
    del _idx
del _eval_only_climates


In [ ]:
# ── Raw input variable standardization (mean/std fit on historical TRAIN only) ─
hist_train_idx = ae_split_indices["historical"]["train"]
X_hist_train_raw = np.asarray(features_by_climate["historical"][hist_train_idx], dtype=np.float32)

input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds = np.ones(n_input_variables, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(var_idx * grid_points_per_patch, (var_idx + 1) * grid_points_per_patch)
    values = X_hist_train_raw[:, cols_for_var].reshape(-1)
    values = values[np.isfinite(values)]
    if var_name == "pr":
        values = np.log1p(values * 86400)
    mu, sigma = float(np.mean(values)), float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0
    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx] = sigma


def standardize_input_variables(X_raw):
    X_raw = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(var_idx * grid_points_per_patch, (var_idx + 1) * grid_points_per_patch)
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = (
            (values - input_variable_means[var_idx]) / input_variable_stds[var_idx]
        ).astype(np.float32)
    return X_scaled


raw_scaled_features_by_climate = {c: standardize_input_variables(features_by_climate[c]) for c in climate_order}
del X_hist_train_raw

print("Raw 15-variable standardization done. Shapes:")
for c in climate_order:
    print(f"  {c}: {raw_scaled_features_by_climate[c].shape}")


## Part 3 - Metadata alignment

Latent files may store samples in a different order than the raw data's test
split. We verify this per (latent setup, climate) using the compound key
`(time, patch_id)`, and build a reindexing array wherever the order differs
-- identical procedure to the original notebook.


In [ ]:
def _sample_keys(meta_df):
    """Return an array of unique 'time|patch_id' string keys, one per row."""
    return (meta_df["time"].astype(str) + "|" + meta_df["patch_id"].astype(str)).values


reindex_maps = {}
_any_reindex = False
for _lkey in latent_test_sets.keys():
    reindex_maps[_lkey] = {}
    for c in climate_order:
        _lmeta = pd.DataFrame(latent_test_sets[_lkey]["latent_test_metadata_by_climate"][c])
        keys_latent = _sample_keys(_lmeta)
        if c == "historical":
            _test_idx = ae_split_indices["historical"]["test"]
            _raw_meta = metadata_by_climate["historical"].iloc[_test_idx].reset_index(drop=True)
        else:
            _raw_meta = metadata_by_climate[c]
        keys_raw = _sample_keys(_raw_meta)

        if len(keys_latent) != len(keys_raw):
            raise ValueError(
                f"[{_lkey}/{c}] Size mismatch: latent has {len(keys_latent)} samples, "
                f"raw test split has {len(keys_raw)}."
            )

        if np.array_equal(keys_latent, keys_raw):
            reindex_maps[_lkey][c] = None
        else:
            _raw_pos = {k: i for i, k in enumerate(keys_raw)}
            _missing = [k for k in keys_latent if k not in _raw_pos]
            if _missing:
                raise ValueError(f"[{_lkey}/{c}] {len(_missing)} latent samples missing from raw test split.")
            reindex_maps[_lkey][c] = np.array([_raw_pos[k] for k in keys_latent], dtype=np.intp)
            _any_reindex = True
            print(f"  Reindex needed for [{_lkey}/{c}]")

if not _any_reindex:
    print("Metadata alignment verified for all 4 setups — no reindexing needed.")


## Part 4 - Cross-climate pairs

For a fixed pool of **10 000 historical anchors** (same target size and same
seed convention as `CMIP_analysis_latent_invariant.ipynb`, so the anchor set
matches), we build 5 pairings:

- **RawData** : nearest neighbour in a 15-D summary space (patch-mean of each
  raw variable) — the "trivial matching" baseline the whole ratio is judged
  against.
- **CERA, Baseline, SWDN, ClimaX** : nearest neighbour in that setup's own
  latent space (48D for the first three, 64D for ClimaX).


In [ ]:
ssp_climates = [c for c in climate_order if c != "historical"]

N_PAIRS_TARGET = 10000
PAIR_SEED = random_seed + 500
_pair_rng = np.random.default_rng(PAIR_SEED)

hist_test_idx = ae_split_indices["historical"]["test"]
hist_meta_test = metadata_by_climate["historical"].iloc[hist_test_idx].reset_index(drop=True)
hist_keys_all = _sample_keys(hist_meta_test)
n_avail_hist = len(hist_keys_all)
N_PAIRS = min(N_PAIRS_TARGET, n_avail_hist)

anchor_pos = _pair_rng.choice(n_avail_hist, size=N_PAIRS, replace=False)
anchor_keys = hist_keys_all[anchor_pos]
anchor_hist_idx = hist_test_idx[anchor_pos]

print(f"Anchor pool: {n_avail_hist} historical test points available -> using {N_PAIRS} anchors "
      f"(target was {N_PAIRS_TARGET}).")


In [ ]:
# ── RawData baseline: nearest neighbour in a 15-D patch-mean summary space ─────
def _patch_mean_summary(flat_block, n_vars, gpp):
    N = flat_block.shape[0]
    block = np.asarray(flat_block, dtype=np.float64).reshape(N, n_vars, gpp)
    return block.mean(axis=2).astype(np.float32)   # (N, n_vars)


def _raw_input_summary(climate):
    X_flat = raw_scaled_features_by_climate[climate]
    if climate == "historical":
        X_flat = X_flat[ae_split_indices["historical"]["test"]]
    return _patch_mean_summary(X_flat, n_input_variables, grid_points_per_patch)


raw_hist_summary_all = _raw_input_summary("historical")
raw_hist_summary_anchor = raw_hist_summary_all[anchor_pos]

raw_ssp_summary_parts, raw_ssp_climate_parts, raw_ssp_index_parts = [], [], []
for c in ssp_climates:
    S = _raw_input_summary(c)
    raw_ssp_summary_parts.append(S)
    raw_ssp_climate_parts.append(np.full(len(S), c))
    raw_ssp_index_parts.append(np.arange(len(S)))

raw_ssp_summary_pool = np.vstack(raw_ssp_summary_parts)
raw_ssp_pool_climate = np.concatenate(raw_ssp_climate_parts)
raw_ssp_pool_index = np.concatenate(raw_ssp_index_parts)

nn_raw = NearestNeighbors(n_neighbors=1).fit(raw_ssp_summary_pool)
_, raw_nn_pos = nn_raw.kneighbors(raw_hist_summary_anchor)
raw_nn_pos = raw_nn_pos[:, 0]

pair_results = {"RawData": (raw_ssp_pool_climate[raw_nn_pos], raw_ssp_pool_index[raw_nn_pos])}
print(f"RawData pairing built ({len(pair_results['RawData'][0])} pairs).")


In [ ]:
# ── The 4 latent-based nearest-neighbour pairings ─────────────────────────────
def build_latent_pairs(latent_key, slice_dim):
    hist_lmeta = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"]["historical"])
    hist_lkeys = _sample_keys(hist_lmeta)
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    missing = [k for k in anchor_keys if k not in key_to_row]
    if missing:
        raise ValueError(f"[{latent_key}] {len(missing)} anchor keys missing from its historical latent block.")
    hist_rows = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"]["historical"])
    Z_hist = Z_hist_all[hist_rows]
    if slice_dim is not None:
        Z_hist = Z_hist[:, :slice_dim]

    Z_ssp_parts, ssp_pool_climate_parts, ssp_pool_index_parts = [], [], []
    for c in ssp_climates:
        Zc = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"][c])
        if slice_dim is not None:
            Zc = Zc[:, :slice_dim]
        lmeta_c = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"][c])
        keys_c = _sample_keys(lmeta_c)
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        idx_in_raw = np.array([raw_key_to_row[k] for k in keys_c])
        Z_ssp_parts.append(Zc)
        ssp_pool_climate_parts.append(np.full(len(Zc), c))
        ssp_pool_index_parts.append(idx_in_raw)

    Z_ssp_pool = np.vstack(Z_ssp_parts)
    ssp_pool_climate = np.concatenate(ssp_pool_climate_parts)
    ssp_pool_index = np.concatenate(ssp_pool_index_parts)

    nn = NearestNeighbors(n_neighbors=1).fit(Z_ssp_pool)
    _, nn_pos = nn.kneighbors(Z_hist)
    nn_pos = nn_pos[:, 0]

    return ssp_pool_climate[nn_pos], ssp_pool_index[nn_pos]


for setup_name in SUBSPACE_SETUPS:
    latent_key, slice_dim = PAIR_LATENT_KEYS[setup_name]
    print(f"Building nearest-neighbour pairs for {setup_name} ({latent_key}, dim={slice_dim or 'full'})...")
    pair_results[setup_name] = build_latent_pairs(latent_key, slice_dim)

print("\nPairs built:")
for setup_name, (tclim, sidx) in pair_results.items():
    print(f"  {setup_name:10s}: {len(tclim)} pairs")


## Part 5 - Standardizing each setup's latent

For each setup, we reconstruct `Z_hist` (the 10 000 anchors) and `Z_ssp_own`
(their match under that setup's *own* pairing), then standardize every
latent coordinate (subtract mean, divide by std, fit on the pooled
hist+ssp population) and compute its covariance matrix `M_pop`.

**No train/holdout split is needed in this notebook.** Parts 6-8 of
`CMIP_analysis_latent_invariant.ipynb` split train/holdout because they
*fit* something to the data (an eigenvector, a symbolic-regression formula)
and needed to check it wasn't overfit noise. Here, the directions we will
score are drawn **randomly**, independently of the data -- there is nothing
to overfit, so we use the full population directly for both computing
`M_pop` and scoring.

`M_pop` will be used to normalize every random direction so that
`Var(v.Z) = 1` for all of them -- this is what makes scores comparable
across directions and across setups regardless of each latent coordinate's
raw scale.


In [ ]:
def get_standardized_Z_for_pairing(setup_name, target_climate, ssp_index, mean=None, std=None):
    """Z_hist, Z_ssp for an arbitrary (target_climate, ssp_index) pairing,
    using setup_name's own latent_key/slice_dim. If mean/std are given,
    returns standardized versions; otherwise returns the raw latent vectors."""
    latent_key, slice_dim = PAIR_LATENT_KEYS[setup_name]
    hist_lmeta = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"]["historical"])
    hist_lkeys = _sample_keys(hist_lmeta)
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    hist_rows = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"]["historical"])
    Z_hist = Z_hist_all[hist_rows]

    Z_ssp = np.empty_like(Z_hist)
    for c in np.unique(target_climate):
        mask = target_climate == c
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        lmeta_c = pd.DataFrame(latent_test_sets[latent_key]["latent_test_metadata_by_climate"][c])
        keys_c = _sample_keys(lmeta_c)
        raw_row_to_latent_row = {raw_key_to_row[k]: i for i, k in enumerate(keys_c)}
        Zc_all = np.asarray(latent_test_sets[latent_key]["latent_test_by_climate"][c])
        latent_rows = np.array([raw_row_to_latent_row[i] for i in ssp_index[mask]])
        Z_ssp[mask] = Zc_all[latent_rows]

    if slice_dim is not None:
        Z_hist = Z_hist[:, :slice_dim]
        Z_ssp = Z_ssp[:, :slice_dim]

    if mean is None:
        return Z_hist, Z_ssp
    return (Z_hist - mean) / std, (Z_ssp - mean) / std


In [ ]:
RATIO_RIDGE_EPS = 1e-6   # numerical stabiliser for M_pop, same role as RIDGE_EPS elsewhere in the project

latent_stats = {}   # {setup_name: {"mean", "std", "M_pop", "dim"}}

for setup_name in SUBSPACE_SETUPS:
    Z_hist, Z_ssp_own = get_standardized_Z_for_pairing(setup_name, *pair_results[setup_name])
    dim = Z_hist.shape[1]

    pool = np.vstack([Z_hist, Z_ssp_own])
    mean = pool.mean(axis=0)
    std = pool.std(axis=0)
    std_safe = np.where(std > 0, std, 1.0)

    Zh_std = (Z_hist - mean) / std_safe
    Zs_std = (Z_ssp_own - mean) / std_safe
    pool_std = np.vstack([Zh_std, Zs_std])
    M_pop = np.cov(pool_std, rowvar=False) + RATIO_RIDGE_EPS * np.eye(dim)

    latent_stats[setup_name] = {"mean": mean, "std": std_safe, "M_pop": M_pop, "dim": dim}
    print(f"{setup_name:10s}: dim={dim}")


## Part 6 - The ratio, on K random directions

### The core score: `Var(g) / L_pair(g)`

For any scalar function of the latent `g(x) = v . Z(x)` (a linear direction
`v`), define:

$$
\text{score}(v) = \frac{\mathrm{Var}(g)}{L_{pair}(g) + \varepsilon},
\qquad
L_{pair}(g) = \mathbb{E}\big[(g(x_{hist}) - g(x_{ssp}))^2\big]
$$

- **`Var(g)`** is computed over the *entire pooled population* involved in a
  pairing (10 000 historical anchors + their 10 000 matched partners, 20 000
  values total) — it asks "does `g` vary meaningfully from one sample to the
  next, across the whole diversity of the data?" A constant `g` has `Var=0`
  and is useless, no matter how "invariant" it looks.
- **`L_pair(g)`** is computed only over *matched pairs* — it asks "does `g`
  stay almost the same for this specific historical patch and the SSP patch
  the network considers its analogue?"

We want both at once: `g` should vary a lot in general (informative), but
change very little between two samples the network has flagged as
physically analogous (invariant). This is exactly the Slow Feature Analysis
idea already used in `CMIP_analysis_latent_invariant.ipynb`'s cell 6-0 and
Part 7 — applied here to *random* directions instead of the single optimal
one.

The ratio is scale-invariant (rescaling `v` by any constant leaves the score
unchanged), so `v` must be normalized some other way to make scores
comparable across directions — see below.

### The "own vs. RawData" ratio

For a given direction `v`, we compute the score twice, using the *same*
historical anchors and the *same* `v`, but two different partners for each
anchor:

- **`score_own(v)`**: partner = this setup's own nearest-neighbour match
  (Part 4).
- **`score_raw(v)`**: partner = the RawData nearest-neighbour match (a
  trivial pairing built with no learned structure at all, from patch-mean
  raw variables).

$$
\text{ratio}(v) = \frac{\text{score}_{own}(v)}{\text{score}_{raw}(v)}
$$

`ratio(v) > 1` means: along direction `v`, the setup's own matching preserves
this quantity *better* than a trivial raw-variable match would — evidence
this specific slice of the latent carries real, non-trivial structure, not
just something RawData's naive proximity would already give away for free.

### Why K *random* directions instead of the single best one

`CMIP_analysis_latent_invariant.ipynb`'s Part 7 solved for the single BEST
direction (via a generalized eigenvalue decomposition) — the exact global
optimum, by construction. That answers "does at least one excellent
invariant axis exist?", but says nothing about the *rest* of the latent: a
setup could have one spectacular axis and be otherwise unremarkable
everywhere else (this is in fact what was found for SWDN).

Sampling **K random directions** instead of the optimal one asks a different
question: **is invariant structure pervasive throughout the whole latent, or
concentrated in a narrow, special corner of it?** If a setup's whole
representation is well-organized physically, a *typical*, unoptimized
direction should already show `ratio > 1` — not just the one direction that
was explicitly searched for.

### Direction normalization

Random directions are drawn as isotropic Gaussian vectors, then rescaled so
that `v.T @ M_pop @ v = 1` (i.e. `Var(v.Z) = 1` over the pooled population) —
the same normalization convention the generalized eigenvectors in Part 7
satisfy automatically. This removes any raw-scale artefact from the
comparison: every direction, in every setup, is put on the same footing
before its `L_pair` is measured.


In [ ]:
def sample_random_Mpop_unit_directions(M_pop, dim, k, seed=0):
    """K random directions, each rescaled so v.T @ M_pop @ v = 1."""
    rng = np.random.default_rng(seed)
    raw = rng.normal(size=(k, dim))
    quad = np.einsum('ij,ij->i', raw, raw @ M_pop)
    return raw / np.sqrt(quad)[:, None]


def score_from_g(gh, gs):
    l_pair = np.mean((gh - gs) ** 2)
    var_g = np.var(np.concatenate([gh, gs]))
    return var_g / (l_pair + 1e-9)


In [ ]:
K_RANDOM_DIRECTIONS = 300
RANDOM_DIR_SEED = 42

ratio_results = {}   # {setup_name: np.ndarray of K ratios}

for setup_name in SUBSPACE_SETUPS:
    mean = latent_stats[setup_name]["mean"]
    std  = latent_stats[setup_name]["std"]
    M_pop = latent_stats[setup_name]["M_pop"]
    dim  = latent_stats[setup_name]["dim"]

    Zh_own, Zs_own = get_standardized_Z_for_pairing(setup_name, *pair_results[setup_name], mean, std)
    Zh_raw, Zs_raw = get_standardized_Z_for_pairing(setup_name, *pair_results["RawData"], mean, std)
    # Zh_own and Zh_raw are identical (same 10,000 anchors, same standardization) -- Zh_own used throughout.

    V = sample_random_Mpop_unit_directions(M_pop, dim, K_RANDOM_DIRECTIONS, seed=RANDOM_DIR_SEED)   # (K, dim)

    Gh = Zh_own @ V.T       # (N_PAIRS, K)
    Gs_own = Zs_own @ V.T
    Gs_raw = Zs_raw @ V.T

    ratios = np.empty(K_RANDOM_DIRECTIONS)
    for k in range(K_RANDOM_DIRECTIONS):
        s_own = score_from_g(Gh[:, k], Gs_own[:, k])
        s_raw = score_from_g(Gh[:, k], Gs_raw[:, k])
        ratios[k] = s_own / s_raw

    ratio_results[setup_name] = ratios
    print(f"{setup_name:10s}: median ratio={np.median(ratios):6.3f}   mean ratio={np.mean(ratios):6.3f}   "
          f"%%>1={100*np.mean(ratios > 1):5.1f}%%")


### Results table and distribution


In [ ]:
summary_rows = []
for setup_name in SUBSPACE_SETUPS:
    r = ratio_results[setup_name]
    summary_rows.append({
        "setup": setup_name,
        "latent dim used": latent_stats[setup_name]["dim"],
        "median ratio": np.median(r),
        "mean ratio": np.mean(r),
        "% directions with ratio > 1": 100 * np.mean(r > 1),
    })

summary_df = pd.DataFrame(summary_rows).set_index("setup")
display(
    summary_df.style
    .format({"median ratio": "{:.2f}", "mean ratio": "{:.2f}", "% directions with ratio > 1": "{:.1f}%"})
    .background_gradient(subset=["median ratio"], cmap="RdYlGn")
    .set_caption(f"Ratio of score_own / score_RawData, aggregated over {K_RANDOM_DIRECTIONS} random directions per setup")
)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

data = [ratio_results[s] for s in SUBSPACE_SETUPS]
bp = ax.boxplot(data, tick_labels=SUBSPACE_SETUPS, showfliers=False, patch_artist=True,
                medianprops=dict(color="black", linewidth=1.5))
for patch in bp["boxes"]:
    patch.set_facecolor("#5B8DB8")
    patch.set_alpha(0.85)

ax.axhline(1.0, color="gray", linestyle="--", linewidth=1.2, label="ratio = 1  (no better than trivial matching)")
ax.set_ylabel("ratio = score_own(v) / score_RawData(v)")
ax.set_title(
    f"Distribution of the ratio over {K_RANDOM_DIRECTIONS} random directions, per setup\n"
    "Higher and more concentrated above 1 = invariant structure is more pervasive across the whole latent"
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.show()


## Part 7 - What this ratio means, and what it does not mean

**What a high aggregate ratio tells you:** across many *unoptimized*,
randomly-chosen linear combinations of this setup's latent coordinates, the
setup's own cross-climate matching keeps that combination more stable than a
trivial nearest-neighbour match on raw variables would. If this holds for
most random directions (not just one cherry-picked axis), it means the
network's geometry is broadly, not narrowly, organized around whatever
notion of "these two climate states are analogous" its training produced.
That is a meaningfully stronger claim than the single-best-direction ratio
computed in `CMIP_analysis_latent_invariant.ipynb`'s Part 7, precisely
because it is not vulnerable to a "one lucky axis" effect.

**What it does NOT tell you:**

- **It does not confirm the structure is genuine atmospheric physics.** A
  pervasively high ratio shows the latent's geometry is broadly consistent
  with the pairing criterion used to build it — it does not rule out that
  this consistency reflects some other systematic property of the dataset
  or training procedure (seasonal cycle, sampling scheme, scenario-specific
  artefacts) rather than a physical conservation law. The correlation-based
  checks against the 25 known physical invariants (done for the top
  eigenvector in `CMIP_analysis_latent_invariant.ipynb`'s Part 7-7) remain
  the way to partially validate this, and were not repeated here for random
  directions.
- **It does not equalize the setups' architectures.** CERA, Baseline, and
  SWDN are variants of a similar encoder family; ClimaX is presumably a much
  larger, differently-designed backbone. A model with more capacity could
  organize *anything* — physical or not — more sharply than a smaller one,
  independent of whether it "understands" the physics better in any
  meaningful sense.
- **Dimensionality is handled differently here than for the top-eigenvalue
  ratio.** For the single best direction, searching a larger space (ClimaX's
  64D vs. 48D for the others) mechanically inflates the achievable optimum
  (more free parameters to fit to sampling noise) — a real confound there.
  For a *random* direction, more dimensions generally *dilutes* a fixed
  amount of signal rather than inflating it, so this metric does not favour
  ClimaX for the same mechanical reason; if ClimaX still comes out ahead
  here, that is not explained by dimensionality alone.
- **No formal uncertainty quantification beyond the empirical spread over
  K draws.** The boxplot above shows the empirical distribution across 300
  random directions per setup (a fixed seed), which is informative about
  spread, but this notebook does not bootstrap over different anchor
  samples or different pairing reconstructions — the anchors and the 4
  pairings themselves are a single fixed realization, exactly as in
  `CMIP_analysis_latent_invariant.ipynb`.

**Bottom line:** treat this ratio as evidence of how *pervasively* a setup's
latent organizes cross-climate structure relative to trivial raw-variable
matching — a genuinely different and complementary question to "does at
least one excellent invariant direction exist" (answered by the top
eigenvector in the companion notebook) — not as a validated, comprehensive
score of "how well this setup has learned atmospheric physics."
